 <font size="10">**Notebook 01: Anndata object generation**</font>
***

<div class="alert alert-info">
    
<b> <h1> ℹ️ <strong> <font size="6" color="black"> Important notebook information </font> </strong> </h1> </b>
    <hr>
    <font size="4" color="black">
        The purpose of this notebook is to generate <b>five</b> anndata objects which can then be used in other pre-processing notebooks later <br> <br>
    These anndata objects will hold raw count matrices in .X partition as they will store outputs from cellranger version 6-0-1 <br> <br>
    The following anndata objects will be made: </font> <br> 
    <ol> <font size="3" color="black">
        <li><b>F137</b> : Embryo_1 : Single Cell sequencing outputs and expression data available in the ArrayExpress database under accession number <a href="https://www.ebi.ac.uk/arrayexpress/experiments/E-MTAB-11504/">E-MTAB-11504</a></li>
        <li><b>F147</b> : Embryo_2 : Single Cell sequencing outputs and expression data available in the ArrayExpress database under accession number <a href="https://www.ebi.ac.uk/arrayexpress/experiments/E-MTAB-11520/">E-MTAB-11520</a></li>
        <li><b>F158</b> : Embryo_3 : Single Cell sequencing outputs and expression data available in the ArrayExpress database under accession number <a href="https://www.ebi.ac.uk/arrayexpress/experiments/E-MTAB-11911/">E-MTAB-11911</a></li>
        <li><b>Combined_embryos</b> : Concatenated anndata object of F137, F147 and F158 with lanes</li>
        <li><b>Combined_embryos_suitable</b> : Combined_embryos anndata object with all lanes unsuitable for downstream analysis removed</li>
    </font> 
    </ol> 
    <font size="4" color="black">
    
<details> 
    <summary> <font size="4" color="black"> Lanes to be removed in combined anndata object </font> </summary> <br/>
    <font size="3" color="black"> Two lanes will be removed from the <b>Combined_embryos</b> anndata object due to reasons which are summarised in table below taken from Cellranger web outputs:<br> <br> </font> 
    <table style="margin-top:10px; margin-left:50px;">
        <tr>
            <th>Sample</th>
            <th>Lane to remove</th>
            <th>Spatial_location_replicate</th>
            <th>Reason</th>
        </tr>
        <tr>
            <td>F137</td>
            <td>WS_wEMB10202360</td>
            <td>Section_7_d</td>
            <td>Low Fraction Valid Barcodes 61.6% Ideal > 75%</td>
        </tr>
        <tr>
            <td>F147</td>
            <td>WS_wEMB11031943</td>
            <td>Section_5_c</td>
            <td>Low Number of Cells Detected 72 Ideal > 100</td>
        </tr>
    </table> <br>
    <font size="3" color="black"> The removal of these lanes will result in the generation of the <b>Combined_embryos_suitable</b> anndata object which is suitable for running downstream analysis in following scripts</font> 
</details>

<div class="alert alert-block alert-warning">
    
<b> <h1> ⚠️ <strong> <font size="6" color="black"> WARNING </font> </strong> ⚠️ </h1> </b>
    <hr>
<p style="color:black;">
    
<strong> <font size="4">  This notebook requires a large amount of memory to run as is from top to bottom. </font> </strong> <br> </p>
    <br>
<details> 
    <summary> <font size="4" color="black"> Memory amount </font> </summary> <br/>
    <font size="3" color="black">
        To run this notebook from top to bottom, the reading memory peaks at 175GB. <br> <br>
        Rough memory usage peaks running this notebook from top to bottom to make the individual anndata objects and save them one after another: <br>
        <ul type="disc">
            <li>Samples F137, F147 and F158: 80GB peak memory usage</li>
            <li>Combined all lanes: 161GB peak memory usage</li>
            <li>Combined with only suitable lanes: 175GB peak memory usage</li>
            </ul> <br>
        Note these peaks were monitored using Grafana not Jupyter lab memory bar which suggested the rough peak memory usage was around 160GB
    </font> 
</details>

<br>

<details> 
    <summary> <font size="4" color="black"> Resource availability </font> </summary> <br/>
    <font size="3" color="black">
        The resources used to run this notebook with high reading memory capabilities and monitor the memory usage using Grafana was provided by <a href="https://www.sanger.ac.uk/group/cellular-genetics-informatics/">Cellular Genetics Informatics</a>
    </font>
</details>

<br>
    
<details>
    <summary> <font size="4" color="black"> Options to decrease memory </font> </summary> <br/>
    <font size="3" color="black">
        It is possible to break this notebook up into separate parts which may help with memory consumption. <br> <br>
        Here is an example to try and reduce the required total amount of memory: <br>
        <ol>
            <li>Run this notebook until the first combined anndata object is saved (F137)</li>
            <li>Restart kernel and re-run script but skip over constructing and saving the first anndata object and instead construct and save the second anndata object (F147)</li>
            <li>Repeat step 2 but skip both the first and second anndata objects and proceed to construct and save the third anndata object (F158)</li>
            <li>Once all individual anndata objects are made for each sample (F137, F147 and F158), load in all anndata objects with corresponding names as variables and proceed to combine into one anndata object conatining all lanes and save</li> 
            <li>Restart kernal and load in only the combined anndata object with all lanes then subset out unsuitable lanes for downstream analysis and save</li>
        </ol>
    </font>
</details>

<br>
    
<details>
    <summary> <font size="4" color="black"> Alternative option to notebook 1 </font> </summary> <br/>
    <font size="3" color="black">
        Notebook 1 can be skipped if you wish to use provided resources which have already combined all lanes for each embryo already. <br> <br>
        These resources can be accessed <a href="https://www.website_link_here">here</a> but you will need to construct the anndata object yourself. <br> <br>
        This can be achieved using a premade function in scanpy package: <a href="https://scanpy.readthedocs.io/en/stable/generated/scanpy.read_10x_mtx.html?msclkid=e6c9fd53a84411ec8708898378d557dc">scanpy.read_10x_mtx</a> 
    </font>
</details>
</div>

***
# Import packages

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import os
import sys
import datetime

#to set unique date for outputs
calc_date = datetime.datetime.now()
date=calc_date.strftime('%Y-%m-%d')
#date=calc_date.strftime('%d-%m-%Y')
date = date.replace('-', '')

sc.settings.verbosity = 2

# Note: package versions can be located in dependencies at the bottom of the script

# User interaction block

In [ ]:
### User settings block ###

# Set root directory so all files and outputs can be found within one overarching location
root_directory = '/home/jovyan/mount_farm/nfs/ar32/Whole_embryo/'


# set directory path to Whole_Embryo_utils.py: This python script hold modules / functions which are used within the script 
Whole_Embryo_utils_path = root_directory + "Utlis/"


# set directory paths for each embryo cellranger outputs:
F137_path = root_directory + 'Cell_ranger_outs/F137/'
F147_path = root_directory + 'Cell_ranger_outs/F147/'
F158_path = root_directory + 'Cell_ranger_outs/F158/'


# set directory path to save anndata object outputs:
# Note if you put in a directory which does not exist you can request for the directory to be made if possible
Metadata_out_path = root_directory + 'Outputs/Metadata/Cell_numbers'
Objects_out_path = root_directory + 'Outputs/Objects/Raw_Objects/'


#############################################################################################################################


### Running checking directory paths to Whole_Embryo_utils.py and save directory paths ###

sys.path.append(Whole_Embryo_utils_path)
import Whole_Embryo_utils as eu

print('\033[1m' + 'Checking file path for saving anndata objects:' + '\033[0m', "\n")
eu.save_path(Objects_out_path)

Checking file path for saving anndata objects: 

File path provided exists
To save data to this location use the inputed variable at the start of the file path


***
# Script start

## Generate anndata object of embryo: F137

In [ ]:
adata_list1 = []
print("="*100)
print("")
for i in list(range(36,81)):
    n = 'WS_wEMB102023'+str(i)
    print(f'Data to load: {n}')
    data = eu.load(F137_path + n + '/filtered_feature_bc_matrix/', Method='mtx')
    data.obs['sequencing_lane_ID'] = n
    adata_list1.append(data)
    print('\033[1m' + f'Data {n} loaded successfully. Data shape is {data.shape}' + '\033[0m', "\n")
    print("="*100,"\n")
    del data # to prevent stackoverflow

F137 = sc.AnnData.concatenate(*adata_list1, join = 'inner', batch_key = None, batch_categories = None, index_unique = None)
F137.obs.index = F137.obs.index.str.replace('-1','',regex=True)
F137.obs.index = F137.obs.index + '-' + F137.obs['sequencing_lane_ID']
F137.obs['haniffa_ID'] = 'F137'
F137.var['genome'] = 'GRCh38'
print("","\n",'\033[1m' + 'Embryo Sample F137 made successfully!' + '\033[0m',"\n")


Data to load: WS_wEMB10202336
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data loaded in successfully
Overall shape: (2068, 36601)
Min count: 0.0
Max count: 4105.0

Data WS_wEMB10202336 loaded successfully. Data shape is (2068, 36601) 


Data to load: WS_wEMB10202337
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data loaded in successfully
Overall shape: (9303, 36601)
Min count: 0.0
Max count: 6596.0

Data WS_wEMB10202337 loaded successfully. Data shape is (9303, 36601) 


Data to load: WS_wEMB10202338
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data loaded in successfully
Overall shape: (9380, 36601)
Min count: 0.0
Max count: 11050.0

Data WS_wEMB10202338 loaded successfully. Data shape is (9380, 36601) 


Data to load: WS_wEMB10202339
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data l

Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
Observation names are not unique. To make them unique, call `.obs_names_make_unique`.


 
 Embryo Sample F137 made successfully! 



## Check anndata object

In [ ]:
data = F137
data_name = 'F137'

print('')
print('\033[1m' + f'{data_name}:'+ '\033[0m' ,data, "\n", "\n")
eu.describe_basic(data)
print('', "\n")
print('\033[1m' + f'{data_name}.obs :' + '\033[0m')
display(data.obs)
print('')
print('\033[1m' + f'{data_name}.var :' + '\033[0m')
display(data.var)


F137: AnnData object with n_obs × n_vars = 330792 × 36601
    obs: 'sequencing_lane_ID', 'haniffa_ID'
    var: 'gene_ids', 'feature_types', 'genome' 
 

Overall shape: (330792, 36601)
Min count: 0.0
Max count: 16057.0
 

F137.obs :


,sequencing_lane_ID,haniffa_ID
AAACCTGAGATCCGAG-WS_wEMB10202336,WS_wEMB10202336,F137
AAACCTGAGTACGATA-WS_wEMB10202336,WS_wEMB10202336,F137
AAACCTGCAGGTCCAC-WS_wEMB10202336,WS_wEMB10202336,F137
AAACCTGGTACCTACA-WS_wEMB10202336,WS_wEMB10202336,F137
AAACCTGGTATTCTCT-WS_wEMB10202336,WS_wEMB10202336,F137
...,...,...
TTTGTCAGTGACCAAG-WS_wEMB10202380,WS_wEMB10202380,F137
TTTGTCAGTTCCAACA-WS_wEMB10202380,WS_wEMB10202380,F137
TTTGTCATCAAACAAG-WS_wEMB10202380,WS_wEMB10202380,F137
TTTGTCATCCAATGGT-WS_wEMB10202380,WS_wEMB10202380,F137



F137.var :


,gene_ids,feature_types,genome
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38
FAM138A,ENSG00000237613,Gene Expression,GRCh38
OR4F5,ENSG00000186092,Gene Expression,GRCh38
AL627309.1,ENSG00000238009,Gene Expression,GRCh38
AL627309.3,ENSG00000239945,Gene Expression,GRCh38
...,...,...,...
AC141272.1,ENSG00000277836,Gene Expression,GRCh38
AC023491.2,ENSG00000278633,Gene Expression,GRCh38
AC007325.1,ENSG00000276017,Gene Expression,GRCh38
AC007325.4,ENSG00000278817,Gene Expression,GRCh38


## Save anndata object: Sample F137
- Saving version of anndata object with all lanes for individual embryo
- This will have cellranger raw counts
- This will allow individual investigation of just sample F137 later without needing to load in any other biological sample

In [ ]:
F137.write_h5ad(Objects_out_path + 'F137_all_lanes_raw_counts_' + date + '.h5ad')

... storing 'sequencing_lane_ID' as categorical
... storing 'haniffa_ID' as categorical
... storing 'feature_types' as categorical
... storing 'genome' as categorical


## Generate anndata object of embryo: F147

In [ ]:
adata_list2 = []
print("="*100)
print("")
for i in list(range(19,67)):
    n = 'WS_wEMB110319'+str(i)
    print(f'Data to load: {n}')
    data = eu.load(F147_path + n + '/filtered_feature_bc_matrix/', Method='mtx')
    data.obs['sequencing_lane_ID'] = n
    adata_list2.append(data)
    print('\033[1m' + f'Data {n} loaded successfully. Data shape is {data.shape}' + '\033[0m', "\n")
    print("="*100,"\n")
    del data # to prevent stackoverflow

F147 = sc.AnnData.concatenate(*adata_list2, join = 'inner', batch_key = None, batch_categories = None, index_unique = None)
F147.obs.index = F147.obs.index.str.replace('-1','',regex=True)
F147.obs.index = F147.obs.index + '-' + F147.obs['sequencing_lane_ID']
F147.obs['haniffa_ID'] = 'F147'
F147.var['genome'] = 'GRCh38'
print("","\n",'\033[1m' + 'Embryo F147 made successfully!' + '\033[0m',"\n")


Data to load: WS_wEMB11031919
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data loaded in successfully
Overall shape: (9471, 36601)
Min count: 0.0
Max count: 4213.0

Data WS_wEMB11031919 loaded successfully. Data shape is (9471, 36601) 


Data to load: WS_wEMB11031920
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data loaded in successfully
Overall shape: (13750, 36601)
Min count: 0.0
Max count: 4305.0

Data WS_wEMB11031920 loaded successfully. Data shape is (13750, 36601) 


Data to load: WS_wEMB11031921
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data loaded in successfully
Overall shape: (13259, 36601)
Min count: 0.0
Max count: 4848.0

Data WS_wEMB11031921 loaded successfully. Data shape is (13259, 36601) 


Data to load: WS_wEMB11031922
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Dat

Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
Observation names are not unique. To make them unique, call `.obs_names_make_unique`.


 
 Embryo F147 made successfully! 



## Check anndata object

In [ ]:
data = F147
data_name = 'F147'

print('')
print('\033[1m' + f'{data_name}:'+ '\033[0m' ,data, "\n", "\n")
eu.describe_basic(data)
print('', "\n")
print('\033[1m' + f'{data_name}.obs :' + '\033[0m')
display(data.obs)
print('')
print('\033[1m' + f'{data_name}.var :' + '\033[0m')
display(data.var)


F147: AnnData object with n_obs × n_vars = 462676 × 36601
    obs: 'sequencing_lane_ID', 'haniffa_ID'
    var: 'gene_ids', 'feature_types', 'genome' 
 

Overall shape: (462676, 36601)
Min count: 0.0
Max count: 12734.0
 

F147.obs :


,sequencing_lane_ID,haniffa_ID
AAACCTGAGCCAGGAT-WS_wEMB11031919,WS_wEMB11031919,F147
AAACCTGAGGACCACA-WS_wEMB11031919,WS_wEMB11031919,F147
AAACCTGAGGCGACAT-WS_wEMB11031919,WS_wEMB11031919,F147
AAACCTGAGGGATACC-WS_wEMB11031919,WS_wEMB11031919,F147
AAACCTGAGTATGACA-WS_wEMB11031919,WS_wEMB11031919,F147
...,...,...
TTTGTCATCATTTGGG-WS_wEMB11031966,WS_wEMB11031966,F147
TTTGTCATCCAGTAGT-WS_wEMB11031966,WS_wEMB11031966,F147
TTTGTCATCCCGACTT-WS_wEMB11031966,WS_wEMB11031966,F147
TTTGTCATCCGGGTGT-WS_wEMB11031966,WS_wEMB11031966,F147



F147.var :


,gene_ids,feature_types,genome
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38
FAM138A,ENSG00000237613,Gene Expression,GRCh38
OR4F5,ENSG00000186092,Gene Expression,GRCh38
AL627309.1,ENSG00000238009,Gene Expression,GRCh38
AL627309.3,ENSG00000239945,Gene Expression,GRCh38
...,...,...,...
AC141272.1,ENSG00000277836,Gene Expression,GRCh38
AC023491.2,ENSG00000278633,Gene Expression,GRCh38
AC007325.1,ENSG00000276017,Gene Expression,GRCh38
AC007325.4,ENSG00000278817,Gene Expression,GRCh38


## Save anndata object: Sample F147
- Saving version of anndata object with all lanes for individual embryo
- This will have cellranger raw counts
- This will allow individual investigation of just sample F137 later without needing to load in any other biological sample

In [ ]:
F147.write_h5ad(Objects_out_path + 'F147_all_lanes_raw_counts_' + date + '.h5ad')

... storing 'sequencing_lane_ID' as categorical
... storing 'haniffa_ID' as categorical
... storing 'feature_types' as categorical
... storing 'genome' as categorical


## Generate anndata object of embryo: F158

In [ ]:
adata_list3 = []
print("="*100)
print("")
for i in list(range(5,56)):
    if i < 10:   # To account for single digit numbers
        i = '0'+str(i)
    if i in list(range(32,35)):    # to account for gap in numbering in lanes
        continue
    n = 'WS_wEMB121421'+str(i)
    print(f'Data to load: {n}')
    data = eu.load(F158_path + n + '/filtered_feature_bc_matrix/', Method='mtx')
    data.obs['sequencing_lane_ID'] = n
    adata_list3.append(data)
    print('\033[1m' + f'Data {n} loaded successfully. Data shape is {data.shape}' + '\033[0m', "\n")
    print("="*100,"\n")
    del data # to prevent stackoverflow

F158 = sc.AnnData.concatenate(*adata_list3, join = 'inner', batch_key = None, batch_categories = None, index_unique = None)
F158.obs.index = F158.obs.index.str.replace('-1','',regex=True)
F158.obs.index = F158.obs.index + '-' + F158.obs['sequencing_lane_ID']
F158.obs['haniffa_ID'] = 'F158'
F158.var['genome'] = 'GRCh38'
print("","\n",'\033[1m' + 'Embryo F158 made successfully!' + '\033[0m',"\n")


Data to load: WS_wEMB12142105
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data loaded in successfully
Overall shape: (10357, 36601)
Min count: 0.0
Max count: 12414.0

Data WS_wEMB12142105 loaded successfully. Data shape is (10357, 36601) 


Data to load: WS_wEMB12142106
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data loaded in successfully
Overall shape: (10456, 36601)
Min count: 0.0
Max count: 6666.0

Data WS_wEMB12142106 loaded successfully. Data shape is (10456, 36601) 


Data to load: WS_wEMB12142107
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

Data loaded in successfully
Overall shape: (9114, 36601)
Min count: 0.0
Max count: 15643.0

Data WS_wEMB12142107 loaded successfully. Data shape is (9114, 36601) 


Data to load: WS_wEMB12142108
Beginning to load data:
Method chosen for loading is mtx out of the options: h5ad , h5 or mtx

D

Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
Observation names are not unique. To make them unique, call `.obs_names_make_unique`.


 
 Embryo F147 made successfully! 



## Check anndata object

In [ ]:
data = F158
data_name = 'F158'

print('')
print('\033[1m' + f'{data_name}:'+ '\033[0m' ,data, "\n", "\n")
eu.describe_basic(data)
print('', "\n")
print('\033[1m' + f'{data_name}.obs :' + '\033[0m')
display(data.obs)
print('')
print('\033[1m' + f'{data_name}.var :' + '\033[0m')
display(data.var)


F158: AnnData object with n_obs × n_vars = 406519 × 36601
    obs: 'sequencing_lane_ID', 'haniffa_ID'
    var: 'gene_ids', 'feature_types', 'genome' 
 

Overall shape: (406519, 36601)
Min count: 0.0
Max count: 15643.0
 

F158.obs :


,sequencing_lane_ID,haniffa_ID
AAACCTGAGACCCACC-WS_wEMB12142105,WS_wEMB12142105,F158
AAACCTGAGATATACG-WS_wEMB12142105,WS_wEMB12142105,F158
AAACCTGAGCTAACAA-WS_wEMB12142105,WS_wEMB12142105,F158
AAACCTGAGGCGCTCT-WS_wEMB12142105,WS_wEMB12142105,F158
AAACCTGAGGGCTTGA-WS_wEMB12142105,WS_wEMB12142105,F158
...,...,...
TTTGTCATCAGCATGT-WS_wEMB12142155,WS_wEMB12142155,F158
TTTGTCATCAGTCCCT-WS_wEMB12142155,WS_wEMB12142155,F158
TTTGTCATCCCTTGCA-WS_wEMB12142155,WS_wEMB12142155,F158
TTTGTCATCCTCATTA-WS_wEMB12142155,WS_wEMB12142155,F158



F158.var :


,gene_ids,feature_types,genome
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38
FAM138A,ENSG00000237613,Gene Expression,GRCh38
OR4F5,ENSG00000186092,Gene Expression,GRCh38
AL627309.1,ENSG00000238009,Gene Expression,GRCh38
AL627309.3,ENSG00000239945,Gene Expression,GRCh38
...,...,...,...
AC141272.1,ENSG00000277836,Gene Expression,GRCh38
AC023491.2,ENSG00000278633,Gene Expression,GRCh38
AC007325.1,ENSG00000276017,Gene Expression,GRCh38
AC007325.4,ENSG00000278817,Gene Expression,GRCh38


## Save anndata object: Sample F158
- Saving version of anndata object with all lanes for individual embryo
- This will have cellranger raw counts
- This will allow individual investigation of just sample F137 later without needing to load in any other biological sample

In [ ]:
F158.write_h5ad(Objects_out_path + 'F158_all_lanes_raw_counts_' + date + '.h5ad')

... storing 'sequencing_lane_ID' as categorical
... storing 'haniffa_ID' as categorical
... storing 'feature_types' as categorical
... storing 'genome' as categorical


# Concatenate all samples together
- Will create a combined anndata object of all 3 embryos into 1 anndata object
- This will have cellranger raw counts
- Save anndata object so have 1 object with all embryos within with no lanes filtered out

In [ ]:
adata_list_all = [F137,F147,F158] 
Combined_embryos = sc.AnnData.concatenate(*adata_list_all, join='inner', batch_key = None, batch_categories = None, index_unique = None)
Combined_embryos.write_h5ad(Objects_out_path + 'Combined_Embryos_all_lanes_raw_counts_' + date + '.h5ad')

... storing 'sequencing_lane_ID' as categorical
... storing 'haniffa_ID' as categorical


## Remove lanes unsuitable for downstream analysis
- Reasonings for lane removal provided above in script 

In [ ]:
# Make list of lanes to remove from combined 
Lanes_to_remove = [
# Embryo1
'WS_wEMB10202360',
# Embryo2
'WS_wEMB11031943'
]

len_cells_before_removal = len(Combined_embryos.obs)
Combined_embryos_suitable = Combined_embryos[~Combined_embryos.obs['sequencing_lane_ID'].isin(Lanes_to_remove),:][:]
len_cells_after_removal = len(Combined_embryos_suitable.obs)
print(f'Number of cells removed by removing unsuitable lanes: {len_cells_before_removal-len_cells_after_removal}', '\n')

data = Combined_embryos_suitable
data_name = 'Combined_embryos_suitable'

print('\033[1m' + f'Final summary of {data_name} anndata object:' + '\033[0m')
eu.describe_basic(data)
print('', "\n")
print('\033[1m' + f'{data_name}.obs :' + '\033[0m')
display(data.obs)
print('')
print('\033[1m' + f'{data_name}.var :' + '\033[0m')
display(data.var)

Number of cells removed by removing unsuitable lanes: 16906 

Final summary of Combined_embryos_suitable anndata object:
Overall shape: (1183081, 36601)
Min count: 0.0
Max count: 16057.0
 

Combined_embryos_suitable.obs :


,sequencing_lane_ID,haniffa_ID
AAACCTGAGATCCGAG-WS_wEMB10202336,WS_wEMB10202336,F137
AAACCTGAGTACGATA-WS_wEMB10202336,WS_wEMB10202336,F137
AAACCTGCAGGTCCAC-WS_wEMB10202336,WS_wEMB10202336,F137
AAACCTGGTACCTACA-WS_wEMB10202336,WS_wEMB10202336,F137
AAACCTGGTATTCTCT-WS_wEMB10202336,WS_wEMB10202336,F137
...,...,...
TTTGTCATCAGCATGT-WS_wEMB12142155,WS_wEMB12142155,F158
TTTGTCATCAGTCCCT-WS_wEMB12142155,WS_wEMB12142155,F158
TTTGTCATCCCTTGCA-WS_wEMB12142155,WS_wEMB12142155,F158
TTTGTCATCCTCATTA-WS_wEMB12142155,WS_wEMB12142155,F158



Combined_embryos_suitable.var :


,gene_ids,feature_types,genome
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38
FAM138A,ENSG00000237613,Gene Expression,GRCh38
OR4F5,ENSG00000186092,Gene Expression,GRCh38
AL627309.1,ENSG00000238009,Gene Expression,GRCh38
AL627309.3,ENSG00000239945,Gene Expression,GRCh38
...,...,...,...
AC141272.1,ENSG00000277836,Gene Expression,GRCh38
AC023491.2,ENSG00000278633,Gene Expression,GRCh38
AC007325.1,ENSG00000276017,Gene Expression,GRCh38
AC007325.4,ENSG00000278817,Gene Expression,GRCh38


# Review metadata
- This will only be done for the Combined_embryos_suitable anndata object as this is the object which will be used for downstream analysis as only contains suitable lanes of interest

In [ ]:
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 150)

print('\033[1m' +"Column dtypes:"+ '\033[0m')
display(Combined_embryos_suitable.obs.info())
print("","\n")
print('\033[1m' +"Number of cells grouped by: embryo -> sequencing lane:"+ '\033[0m')
display(Combined_embryos_suitable.obs.groupby(['haniffa_ID','sequencing_lane_ID']).apply(len))
Cell_numbers_table = pd.DataFrame(Combined_embryos_suitable.obs.groupby(['haniffa_ID','sequencing_lane_ID']).apply(len))
Cell_numbers_table.rename(columns = {0:'No_Cells'}, inplace = True)
Cell_numbers_table.to_csv(Metadata_out_path + 'Combined_Embryos_all_suitable_lanes_raw_cell_numbers_per_lane_' + date + '.csv')

Column dtypes:
<class 'anndata._core.views.DataFrameView'>
Index: 1183081 entries, AAACCTGAGATCCGAG-WS_wEMB10202336 to TTTGTCATCCTTCAAT-WS_wEMB12142155
Data columns (total 2 columns):
 #   Column              Non-Null Count    Dtype   
---  ------              --------------    -----   
 0   sequencing_lane_ID  1183081 non-null  category
 1   haniffa_ID          1183081 non-null  category
dtypes: category(2)
memory usage: 12.4+ MB


None

 

Number of cells grouped by: embryo -> sequencing lane:


haniffa_ID  sequencing_lane_ID
F137        WS_wEMB10202336        2068
            WS_wEMB10202337        9303
            WS_wEMB10202338        9380
            WS_wEMB10202339        9223
            WS_wEMB10202340        7515
            WS_wEMB10202341        9679
            WS_wEMB10202342       10539
            WS_wEMB10202343        9504
            WS_wEMB10202344        9489
            WS_wEMB10202345        6809
            WS_wEMB10202346        6498
            WS_wEMB10202347        6866
            WS_wEMB10202348        8410
            WS_wEMB10202349        6688
            WS_wEMB10202350        6349
            WS_wEMB10202351        6563
            WS_wEMB10202352        7549
            WS_wEMB10202353        7314
            WS_wEMB10202354        7376
            WS_wEMB10202355        8433
            WS_wEMB10202356        6559
            WS_wEMB10202357        7380
            WS_wEMB10202358        6899
            WS_wEMB10202359        7029
         

## Save anndata object: Combined anndata object
- Saving version of anndata object with all embryos together
- This object has unsuitable lanes for downstream analysis removed
- This object has cellranger raw counts
- This will be used for downstream analysis

In [ ]:
Combined_embryos_suitable.write_h5ad(Objects_out_path + 'Combined_Embryos_all_suitable_lanes_raw_counts_' + date + '.h5ad')

***
### Dependencies
Here are some information about resources and import package versions used within this script

In [1]:
%load_ext watermark
%watermark -v -m -p numpy,pandas,scanpy,anndata

Python implementation: CPython
Python version       : 3.9.4
IPython version      : 7.24.1

numpy  : 1.20.3
pandas : 1.2.4
scanpy : 1.7.2
anndata: 0.7.6

Compiler    : GCC 9.3.0
OS          : Linux
Release     : 4.15.0-112-generic
Machine     : x86_64
Processor   : x86_64
CPU cores   : 26
Architecture: 64bit

